# 🔎 03. Analiza sentymentu
## 🔎 Metody mieszane w analizie tekstu: od słowników do BERT

**Cel:** opisać ton wypowiedzi i porównać grupy z uwzględnieniem niepewności. Sentyment nie jest diagnozą, dobrostanem ani nasileniem objawów. VADER to angielski leksykon z regułami, nie transformer.

# 🧭 Jak wykonać ten notebook

1. Otwórz plik `.ipynb` w Google Colab i zapisz własną kopię na Dysku Google.
2. **Wystarczy CPU.** Komórka to jeden blok tekstu albo kodu. Kod uruchamiasz przyciskiem ▶ po lewej lub Shift+Enter.
3. Uruchom pierwszą komórkę kodu. Jeżeli zainstaluje biblioteki i pokaże 🔄, uruchom ponownie sesję z menu **Środowisko wykonawcze**. Potem zacznij od pierwszej komórki. Restart zachowuje pliki, ale usuwa zmienne z pamięci.
4. Wykonuj kod **od góry, bez pomijania komórek**. Dane pobiorą się automatycznie z [repozytorium prowadzącego](https://github.com/bartlomiejnowak-ux/PSPS-2026). Domyślnie niczego nie wgrywasz. W razie awarii pobierania komórka podaje instrukcję ręcznego wgrania.
5. Poczekaj, aż obracający się znacznik przy komórce zniknie. Pierwsze pobranie modelu i obliczenia mogą potrwać kilka minut, a BERTopic dłużej. Nie klikaj wielokrotnie ▶.
6. ✅ oznacza sukces, 🔎 wskazuje co przeczytać, ⚠️ ważne ograniczenie, ✏️ ćwiczenie. Emoji nie zmieniają działania kodu.
7. Na końcu pobierz ZIP wyników. Sam zapis notebooka na Dysku nie zachowuje plików z tymczasowej sesji.


### 🛠️ Gdy coś nie działa

| Objaw | Co zrobić |
|---|---|
| `NameError` lub „nie zdefiniowano” | Pominięto wcześniejszy krok albo zrestartowano sesję. Wykonaj kod od początku. |
| `ModuleNotFoundError` / błąd wersji biblioteki | Uruchom instalację, zrestartuj sesję i wykonaj komórki od góry. |
| Brak pliku / zła kolumna | Ponów komórkę pobierania danych. Awaryjnie wybierz `cleaned_topic_modeling_dataset(1).csv` z repozytorium. |
| Błąd pobierania modelu | Sprawdź połączenie, zaczekaj i ponów komórkę pobierania. Nie zmieniaj nazwy modelu. |
| Błąd po zmianie parametru | Cofnij zmianę lub przywróć wartości pokazane w komentarzach i wykonaj dalsze komórki kolejno. |
| Sesja wygasła | Połącz ponownie, uruchom notebook od początku i ponownie wykonaj komórkę pobierania danych. |

Nie przechodź dalej po czerwonym błędzie. Czytaj ostatnią linijkę komunikatu. W tej kopii wyniki pojawią się dopiero po uruchomieniu kodu. Liczby na slajdach pochodzą ze sprawdzonego wcześniejszego wykonania; przy innych ustawieniach lub środowisku wynik może się różnić.

## 📖 Słowa i skróty używane w tym module

- **VADER:** narzędzie oceny tonu tekstu oparte na słowniku i regułach.
- **pos / neg / neu:** udziały tonu dodatniego / ujemnego / neutralnego według narzędzia.
- **compound:** łączny wynik tonu od −1 do 1.
- **SD:** odchylenie standardowe: rozrzut wyników wokół średniej.
- **95% CI:** przedział ufności; przy powtarzaniu poprawnej procedury około 95% przedziałów obejmie prawdziwy parametr.
- **Welch ANOVA:** test różnic średnich, który dopuszcza nierówne wariancje (rozrzuty).
- **F i p:** F porównuje różnice z szumem; p ocenia zgodność danych z hipotezą braku różnic i założeniami testu, nie prawdopodobieństwo prawdziwości hipotezy.
- **Games–Howell:** porównanie par średnich z korektą za wielokrotne porównania.
- **η² (eta kwadrat):** opisowy udział zmienności związany z podziałem na grupy.
- **ρ (rho) Spearmana:** zgodność porządków wyników, od −1 do 1.
- **Deduplikacja:** usunięcie powtórzonych tekstów.
- **Skala logarytmiczna / log:** ściska duże wartości, żeby długi ogon rozkładu mieścił się na wykresie. Równe odstępy na osi nie oznaczają równych przyrostów liczby słów.

# 🔎 1. Przygotowanie
Uruchom komórkę instalacji poniżej. W Colab wszystko wykonujesz w przeglądarce.

### ▶️ Krok kodu 1

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** ✅ Biblioteki gotowe albo instrukcja jednorazowego restartu 🔄.

In [ ]:
import sys, subprocess, importlib.util, importlib.metadata as metadata
from pathlib import Path
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
PACKAGES = ['spacy==3.8.16', 'https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl', 'numpy==2.5.3', 'pandas==3.0.5', 'matplotlib==3.11.2', 'scipy==1.18.1', 'vaderSentiment==3.3.2']
if sys.version_info < (3, 12):
    raise RuntimeError('⛔ Ten zestaw wersji wymaga Python 3.12 lub nowszego. Wybierz zgodne środowisko.')
def installed(spec):
    name, expected = ('en-core-web-sm', '3.8.0') if spec.startswith('https:') else spec.split('==')
    try: return metadata.version(name) == expected
    except metadata.PackageNotFoundError: return False
needed = [spec for spec in PACKAGES if not installed(spec)]
if needed and IN_COLAB:
    print('⏳ Instalacja bibliotek. Poczekaj na zakończenie tej komórki.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
    raise RuntimeError('🔄 Instalacja zakończona. Uruchom ponownie sesję z menu Środowisko wykonawcze, a następnie wykonaj komórki od początku. To jednorazowy krok po instalacji.')
if needed:
    raise RuntimeError('⛔ Brakuje wymaganych wersji. Lokalnie użyj pliku requirements właściwego modułu. W Colab komórka instaluje je sama. Braki: ' + ', '.join(needed))
print('✅ Biblioteki gotowe. Możesz uruchomić następną komórkę.')


### ▶️ Krok kodu 2

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
MODULE = "03_SENTIMENT_ANALYSIS"
from pathlib import Path
import json, re, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
START = time.perf_counter()
HERE = Path.cwd()
ROOT = HERE.parent if HERE.name.startswith(("01_", "02_", "03_", "04_")) else HERE
(ROOT / "data").mkdir(exist_ok=True)
OUT = ROOT / MODULE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
TARGET = {0:"Stress", 1:"Depression", 2:"Bipolar", 3:"Personality Disorder", 4:"Anxiety"}
RNG = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi":120, "axes.spines.top":False, "axes.spines.right":False})
print("Folder wyników:", OUT)

from scipy import stats
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
except ImportError as exc:
    raise RuntimeError("Zainstaluj vaderSentiment: python -m pip install -r ../requirements_workshop.txt") from exc

# 🔎 2. Wczytanie danych
VADER dostaje **oryginalną kolumnę document**, z negacją, interpunkcją i wielkimi literami zachowanymi w dostarczonych danych.

### ▶️ Krok kodu 3

Uruchom raz i poczekaj. **Oczekiwany efekt:** automatyczne pobranie danych i komunikat ✅. Przy awarii ustaw w kodzie `DATA_SOURCE = 'upload'` i wybierz **`cleaned_topic_modeling_dataset(1).csv`** z repozytorium prowadzącego.

In [ ]:
# 📥 Dane z repozytorium prowadzącego. Domyślnie niczego nie wgrywasz ręcznie.
import io, urllib.request, hashlib
DATA_SOURCE = 'github'  # awaryjnie zmień na 'upload' i uruchom komórkę ponownie
GITHUB_COMMIT = '1cb710169713a4f8ea5a0d839be6d6e1df8ab078'
GITHUB_URL = f'https://raw.githubusercontent.com/bartlomiejnowak-ux/PSPS-2026/{GITHUB_COMMIT}/cleaned_topic_modeling_dataset%281%29.csv'
EXPECTED_SHA256 = '4fb7bdc1779127e875ce3ea7fd571f85885b700937b9439bd606b7c2d67d7972'
INPUT_NAME = 'cleaned_topic_modeling_dataset(1).csv'
if IN_COLAB:
    if DATA_SOURCE == 'github':
        print('⏳ Pobieranie danych z GitHub…')
        try:
            with urllib.request.urlopen(GITHUB_URL, timeout=60) as response:
                data_bytes = response.read()
        except Exception as exc:
            raise RuntimeError('⛔ Nie udało się pobrać danych. Sprawdź internet i ponów tę komórkę. Awaryjnie ustaw DATA_SOURCE = "upload" i wybierz plik z repozytorium prowadzącego.') from exc
        if hashlib.sha256(data_bytes).hexdigest() != EXPECTED_SHA256:
            raise ValueError('⛔ Pobrany plik nie zgadza się ze sprawdzoną wersją. Nie kontynuuj analizy na tym pliku.')
    elif DATA_SOURCE == 'upload':
        from google.colab import files
        print('📂 Wybierz cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('⛔ Wybierz dokładnie jeden plik i ponów tę komórkę.')
        data_bytes = next(iter(uploaded.values()))
    else:
        raise ValueError('⛔ DATA_SOURCE musi mieć wartość "github" albo "upload".')
    try:
        source_df = pd.read_csv(io.BytesIO(data_bytes), keep_default_na=False)
    except Exception as exc:
        raise ValueError('⛔ Nie można odczytać pliku. Wgraj cleaned_topic_modeling_dataset(1).csv z repozytorium prowadzącego.') from exc
    missing_columns = {'target', 'document'} - set(source_df.columns)
    if source_df.empty or missing_columns:
        raise ValueError(f'⛔ Pusty lub niewłaściwy plik. Brakujące kolumny: {sorted(missing_columns)}')
    data_folder = ROOT / 'data'
    data_folder.mkdir(parents=True, exist_ok=True)
    (data_folder / INPUT_NAME).write_bytes(data_bytes)
    print(f'✅ Dane gotowe: {len(source_df)} wierszy. Źródło: {DATA_SOURCE}.')

# Moduły 02–03 odtwarzają przygotowanie z modułu 01 w osobnej sesji Colab.
if IN_COLAB:
    import re, json, spacy
    print('⏳ Przygotowanie tokenów i form podstawowych. Poczekaj kilka minut.')
    source_df.insert(0, 'source_row', np.arange(len(source_df)))
    def clean_for_workshop(text):
        text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
        text = re.sub(r'(?<!\w)(?:u/|@)[A-Za-z0-9_-]+', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()
    source_df['document_clean'] = source_df.document.map(clean_for_workshop)
    source_df['tokens'] = source_df.document_clean.map(lambda t: re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", t.lower()))
    source_df['n_words'] = source_df.tokens.map(len)
    prep_nlp = spacy.load('en_core_web_sm', disable=['parser','ner'])
    all_lemmas, content_lemmas = [], []
    for doc in prep_nlp.pipe(source_df.document_clean.tolist(), batch_size=64):
        all_lemmas.append([t.lemma_.lower() for t in doc if t.is_alpha])
        content_lemmas.append(' '.join(t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop))
    source_df['tokens_lemma'] = all_lemmas
    source_df['document_lemma'] = content_lemmas
    for column in ['tokens','tokens_lemma']:
        source_df[column] = source_df[column].map(lambda values: json.dumps(values, ensure_ascii=False))
    source_df.to_csv(data_folder / 'workshop_preprocessed.csv', index=False)
    print('✅ Przygotowano dane do tego modułu. Nie trzeba wcześniej uruchamiać modułu 01.')


### ▶️ Krok kodu 4

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
input_path = ROOT / "data/workshop_preprocessed.csv"
if not input_path.exists():
    raise FileNotFoundError("Najpierw wykonaj notebook 01: zapisuje data/workshop_preprocessed.csv.")
df = pd.read_csv(input_path, keep_default_na=False)
df["tokens"] = df.tokens.map(json.loads)
df["tokens_lemma"] = df.tokens_lemma.map(json.loads)
assert df.target.isin(TARGET).all() and df.source_row.is_unique
assert np.array_equal(df.n_words, df.tokens.map(len))
df["group"] = df.target.map(TARGET)
display(df[["source_row", "target", "n_words"]].head())

# 🔎 3. Kontrola danych
Porównujemy dokumenty, nie osoby. Nie mamy identyfikatorów autorów ani schematu doboru próby. Testy i CI mają charakter warsztatowy, warunkowy względem niezależności wierszy.

### ▶️ Krok kodu 5

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
display(df.groupby("group").n_words.agg(["size", "mean", "median"]).reindex(TARGET.values()))
print("Powtórzenia tekstu:", df.document.duplicated().sum())

# 🔎 4. Analiza
## 🔎 4.1 Positive, negative, neutral i compound
`pos`, `neg`, `neu` to proporcje przypisywane przez narzędzie, nie prawdopodobieństwa prawdziwych emocji. `compound` jest znormalizowanym wynikiem reguł w zakresie −1…1; nie jest po prostu pos−neg. [Opis autorów VADER](https://github.com/cjhutto/vaderSentiment#about-the-scoring).

### ▶️ Krok kodu 6

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
analyzer = SentimentIntensityAnalyzer()
sentiment = pd.DataFrame(df.document.map(analyzer.polarity_scores).tolist())
sentiment.index = df.index
df = df.drop(columns=list(sentiment.columns), errors="ignore").join(sentiment)
assert df.compound.between(-1, 1).all()
assert np.allclose(df[["pos", "neg", "neu"]].sum(axis=1), 1, atol=.002)
display(df[["source_row", "pos", "neg", "neu", "compound"]].head())
df["sentiment_label"] = np.select([df.compound.ge(.05), df.compound.le(-.05)], ["positive", "negative"], default="neutral")
display(df.sentiment_label.value_counts(normalize=True))

## 🔎 4.2 Opis grup i 95% CI
Bootstrap średnich dokumentów: 1000 prób. CI nie uwzględniają nieznanych autorów ani sposobu doboru korpusu.

### ▶️ Krok kodu 7

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** przygotowanie funkcji lub ustawień do dalszych kroków; brak wydruku jest prawidłowy.

In [ ]:
def mean_ci(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    means = [RNG.choice(values, len(values), replace=True).mean() for _ in range(1000)]
    return pd.Series({"n":len(values), "mean":values.mean(), "sd":values.std(ddof=1),
                      "low":np.quantile(means, .025), "high":np.quantile(means, .975)})

## 🔎 Tabela opisowa
Efekt będziemy oceniać w jednostkach compound, obok SD i mediany.

### ▶️ Krok kodu 8

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
summary = pd.DataFrame([{"target":i, "group":TARGET[i], **mean_ci(g.compound), "median":g.compound.median()}
                         for i,g in df.groupby("target")])
display(summary.round(4))
summary.to_csv(OUT / "sentiment_by_target.csv", index=False)

## 🔎 4.3 Welch ANOVA
Pytanie: czy średnie wyniki w pięciu grupach są równe? Welch dopuszcza nierówne wariancje. Nadal wymaga niezależności i ma charakter przybliżony przy niestandardowych rozkładach. Duże n nie naprawia błędnego pomiaru. [SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html).

### ▶️ Krok kodu 9

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
groups = [df.loc[df.target.eq(i), "compound"].to_numpy() for i in TARGET]
welch = stats.f_oneway(*groups, equal_var=False)
sizes = np.array([len(x) for x in groups]); variances = np.array([x.var(ddof=1) for x in groups])
weights = sizes / variances; k = len(groups)
correction = np.sum((1-weights/weights.sum())**2 / (sizes-1))
df2 = (k*k-1) / (3*correction)
grand_mean = df.compound.mean()
ss_between = sum(len(x)*(x.mean()-grand_mean)**2 for x in groups)
eta_squared_descriptive = ss_between / np.sum((df.compound-grand_mean)**2)
test_summary = {"F_Welch":float(welch.statistic), "df1":k-1, "df2":float(df2), "p":float(welch.pvalue),
                "eta_squared_descriptive":float(eta_squared_descriptive)}
display(pd.Series(test_summary))

## 🔎 4.4 Games–Howell: które średnie się różnią?
W SciPy `tukey_hsd(equal_var=False)` wykonuje Games–Howell. Raportujemy różnicę średnich (efekt w jednostkach compound), jednoczesny 95% CI i p uwzględniające rodzinę 10 porównań. Nie dodajemy drugiej korekty do tych p. η² powyżej jest opisowym udziałem wariancji, a nie estymatorem specyficznym dla testu Welcha. [Dokumentacja](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.tukey_hsd.html).

### ▶️ Krok kodu 10

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
posthoc = stats.tukey_hsd(*groups, equal_var=False)
intervals = posthoc.confidence_interval(.95)
pairwise = pd.DataFrame([{"group_a":TARGET[i], "group_b":TARGET[j],
    "mean_difference":posthoc.statistic[i,j], "ci_low":intervals.low[i,j], "ci_high":intervals.high[i,j],
    "p_Games_Howell":posthoc.pvalue[i,j]} for i in range(5) for j in range(i+1,5)])
display(pairwise.round(5))
pairwise.to_csv(OUT / "games_howell.csv", index=False)

## 🔎 4.5 Wrażliwość na duplikaty
Usuwamy identyczne teksty tylko w analizie pomocniczej. Zachowanie pierwszej etykiety przy konflikcie jest decyzją, którą jawnie raportujemy. Nawet unikalny tekst nie gwarantuje unikalnego autora.

### ▶️ Krok kodu 11

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
conflicts = df.groupby("document").target.nunique().gt(1).sum()
unique = df.drop_duplicates("document", keep="first")
unique_groups = [unique.loc[unique.target.eq(i), "compound"].to_numpy() for i in TARGET]
unique_welch = stats.f_oneway(*unique_groups, equal_var=False)
sensitivity = {"unique_documents":len(unique), "texts_with_conflicting_labels":int(conflicts),
               "F_Welch_unique":float(unique_welch.statistic), "p_unique":float(unique_welch.pvalue)}
display(pd.Series(sensitivity))
display(unique.groupby("target").compound.mean().rename(index=TARGET))

## 🔎 4.6 Składniki dodatnie i ujemne
Szukamy dwóch prawdziwych dokumentów o podobnym compound i różnym udziale sygnałów. Algorytm wybiera przykłady według jawnej reguły, nie według kategorii target.

### ▶️ Krok kodu 12

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
df["compound_bin"] = df.compound.round(1)
df["affective_share"] = df.pos + df.neg
candidate_bin = df.groupby("compound_bin").affective_share.agg(lambda x: x.max()-x.min()).idxmax()
candidates = df[df.compound_bin.eq(candidate_bin)]
composition = df.loc[[candidates.affective_share.idxmin(), candidates.affective_share.idxmax()],
                    ["source_row", "pos", "neg", "neu", "compound", "document"]].copy()
composition["document"] = composition.document.str.slice(0,350)
display(composition)
composition.to_json(OUT / "composition_examples.json", orient="records", force_ascii=False)

# 🔎 5. Wykresy
## 🔎 5.1 Cały korpus
Progi ±0,05 są konwencją VADER; neutralny wynik może oznaczać brak rozpoznanych sygnałów lub równoważące się treści.

### ▶️ Krok kodu 13

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df.compound, bins=40, color="#2B8C8E")
ax.set(xlabel="Compound VADER", ylabel="Dokumenty", title="Sentyment w całym korpusie")
fig.tight_layout(); fig.savefig(OUT / "sentiment_overall.png"); plt.show()

## 🔎 Rozkłady w grupach
Wykresy mogą pokazywać asymetrię i koncentrację przy granicach skali.

### ▶️ Krok kodu 14

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
groups = [df.loc[df.target.eq(i), "compound"].to_numpy() for i in TARGET]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].boxplot(groups, tick_labels=list(TARGET.values()), showfliers=False)
axes[0].tick_params(axis="x", rotation=30); axes[0].set(ylabel="Compound", title="Rozkłady (bez punktów odstających)")
axes[1].errorbar(summary["mean"], range(5), xerr=[summary["mean"]-summary.low, summary.high-summary["mean"]], fmt="o", capsize=3, color="#2B8C8E")
axes[1].set_yticks(range(5), TARGET.values()); axes[1].set(xlabel="Compound", title="Średnie i 95% CI bootstrap")
fig.tight_layout(); fig.savefig(OUT / "sentiment_groups.png"); plt.show()

## 🔎 5.3 Sentyment × długość
Korelacja rang Spearmana opisuje monotoniczną zależność. Długość może też sprzyjać nasyceniu compound; nie wnioskujemy o przyczynowości.

### ▶️ Krok kodu 15

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** wykres.

In [ ]:
length_relation = stats.spearmanr(df.n_words, df.compound)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hexbin(np.log1p(df.n_words), df.compound, gridsize=35, mincnt=1, cmap="viridis")
ax.set(xlabel="log(1 + liczba słów)", ylabel="Compound", title=f"Długość i sentyment: rho = {length_relation.statistic:.3f}")
fig.tight_layout(); fig.savefig(OUT / "sentiment_length.png"); plt.show()
print("Spearman rho, p:", length_relation)

## 🔎 5.4 Przykłady rzeczywistych tekstów
Skrajne wyniki nie są orzeczeniem o stanie autora. Skracamy wyświetlanie do 350 znaków; pełny tekst pozostaje w danych.

### ▶️ Krok kodu 16

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
example_rows = pd.concat([df.nlargest(2,"compound").assign(selection="positive"),
                          df.nsmallest(2,"compound").assign(selection="negative"),
                          df.loc[df.compound.abs().nsmallest(2).index].assign(selection="near-neutral")])
example_rows = example_rows[["source_row","selection","compound","document"]].copy()
example_rows["document"] = example_rows.document.str.slice(0,350)
display(example_rows)
example_rows.to_json(OUT / "sentiment_examples.json", orient="records", force_ascii=False)

# 🔎 6. Interpretacja
## 🔎 Próby obciążeniowe narzędzia
Poniższe przykłady są **skonstruowane**, a nie pobrane z korpusu. VADER często obsługuje prostą negację; trudniejsze zakresy negacji, sarkazm i język kliniczny nadal wymagają oceny człowieka. Bez anotacji nie nazywamy wszystkich rozbieżności zweryfikowanymi błędami.

### ▶️ Krok kodu 17

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** tabela z wynikami lub przykładami.

In [ ]:
stress_tests = [
    ("negacja", "I am not unhappy.", "Podwójna negacja nie jest prostym zaprzeczeniem smutku."),
    ("sarkazm", "Great, another panic attack. Exactly what I needed.", "Pozytywne słowa mogą być ironiczne."),
    ("język kliniczny", "The test was positive for cancer.", "Positive ma znaczenie medyczne."),
    ("mieszane emocje", "I love my family but I hate how lonely I feel.", "Jeden wynik nie opisze obu wątków."),
    ("kontekst", "My therapist said I am doing great, but I do not believe it.", "Cytat i ocena autora są różne.")]
failures = pd.DataFrame([{"case":kind,"text":text,"caution":caution,**analyzer.polarity_scores(text)} for kind,text,caution in stress_tests])
display(failures[["case","text","compound","caution"]])
failures.to_json(OUT / "failure_cases.json", orient="records", force_ascii=False)

## 🔎 Zapis i interpretacja testu
Małe p mówi o niezgodności z modelem równych średnich, a nie o ważnym efekcie. Porównaj różnice, CI, rozkłady i przykład tekstu. Zależność od kategorii może odzwierciedlać język tematyczny, długość i dobór próby.

### ▶️ Krok kodu 18

Uruchom raz i poczekaj na zakończenie. **Oczekiwany efekt:** komunikat lub wyniki tekstowe pod komórką.

In [ ]:
df[["source_row","target","n_words","pos","neg","neu","compound","sentiment_label"]].to_csv(OUT / "sentiment_scores.csv", index=False)
test_summary.update(sensitivity)
test_summary.update({"documents":len(df), "spearman_length_rho":float(length_relation.statistic),
    "spearman_length_p":float(length_relation.pvalue), "runtime_seconds":time.perf_counter()-START})
(OUT / "summary.json").write_text(json.dumps(test_summary, indent=2), encoding="utf-8")

# 🔎 7. Ćwiczenie
1. Przeczytaj po dwa skrajne i neutralne teksty. Czy ton autora odpowiada wynikowi?
2. Wybierz jedną różnicę Games–Howell: opisz jej wielkość i CI bez słowa „istotna”.
3. Zaplanuj próbkę do podwójnego kodowania sentymentu i sposób rozstrzygania rozbieżności.
4. Co zmieniłoby posiadanie identyfikatorów autorów?

## 🔎 Most do modułu 4
| Metoda | Co reprezentuje? | Skąd kategorie? |
|---|---|---|
| Słownik | wystąpienia wybranych słów | definiuje badacz |
| Sentyment | szacowany ton wartościujący | wcześniej zdefiniowany leksykon/model |
| Embeddingi | podobieństwo semantyczne w geometrii | reprezentacja wyuczona wcześniej |
| Topic modeling | grupy podobnych dokumentów i ich opis | klastry odkrywane w korpusie |

PCA i UMAP redukują wymiary, HDBSCAN wyznacza klastry, c-TF-IDF/MMR pomagają je opisać. Target pozostanie wyłącznie kryterium zewnętrznym.

Dwa teksty mogą mieć podobny sentyment i różne znaczenie albo podobne znaczenie i różny sentyment. Poprawny wniosek dotyczy dokumentów w grupie z etykietą Depression, a nie wszystkich osób z depresją.

## 💾 Pobranie wyników

Uruchom tę komórkę po ukończeniu analizy. Utworzy ZIP i w Colab rozpocznie pobieranie. Jeśli przeglądarka je zablokuje, odszukaj ZIP w panelu Pliki i pobierz ręcznie. Zachowaj też własną kopię notebooka.

In [ ]:
import shutil
bundle = Path('wyniki_modul_03')
bundle.mkdir(exist_ok=True)
if not Path(OUT).exists():
    raise RuntimeError('⛔ Najpierw wykonaj komórki analizy i zapisu wyników.')
shutil.copytree(OUT, bundle / 'tabele_i_wykresy', dirs_exist_ok=True)
archive = shutil.make_archive(str(bundle), 'zip', bundle)
print('✅ Plik wyników:', archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
